This notebook will be used to understand how the model runs and the results it gives.

### Annotating README workflow

The README details how to install the environment such that all dependencies are available.

In [ ]:
# 315kV LOPF (0% shed) and DC power flow check
%cd ~/Documents/McGill/capstone/quebec-pypsa-earth #replace with path to your repo!

!python network/run_lopf_main_island.py --network network/networks_current/elec_reduced.nc --output etienne_tests/results/elec_solved.nc
!python network/run_pf.py --network etienne_tests/results/elec_solved.nc --method lpf

First line  
Description: Runs the LOPF on 315 kV network. Gives what must be generated.
Input: 
- run_lopf_main_island.py --> solves the linearised optimal power flow for main grid island. Only main island because secondary islands means they are connected by lower transmission lines.
- network/networks_current/elec_reduced.nc --> Quebec topology. Details grid components.  

Output:
- results/elec_solved.nc --> gives generation, line flows, if there is load shedding

FIX THE PRODUCTION, now how does it flow in the network?  

Second line:
Description: Runs LPF to see if the previously run LOPF is physically feasible on the network.
Input:
- run_pf.py --> runs linear power flow (DC).
- results/elec_solved.nc --> is this generation possible?

Output:
- results/elec_solved_lpf.nc --> bus voltage angles, line flows, transformer flows,


**LOPF**
$$ min\sum_{i \in generators} c_i \times P_{g,i}$$

Minimizes the total generation costs, with the following constraints:

1) At each bus:
$$Generation - Load = Transmission lines$$

$$\implies \sum_{i \in gen at k} P_{g,i} - P_{demand,k} = \sum_{j \in lines} B_{k,j}(\theta_k - \theta_j)$$

where $B_{k,j}$ is the susceptance of the line between the two buses (how easily AC current can flow through).

2) Generator has limits

3) Transmission lines have thermal limits

In [ ]:
# Reduce to 735kV -> AC PF 89/168
%cd ~/Documents/McGill/capstone/quebec-pypsa-earth #replace with path to your repo!
!python network/reduce_to_735kv.py
!python network/run_pf.py --network network/networks_current/elec_735kv.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

/home/etienne/Documents/McGill/capstone/quebec-pypsa-earth
Loading solved network: /home/etienne/Documents/McGill/capstone/quebec-pypsa-earth/network/networks_current/elec_solved.nc
INFO:pypsa.io:Imported network elec_solved.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers
  205 buses, 276 lines, 22 transformers, 5 links, 262 generators, 22 storage units, 647 loads

Backbone: 58 buses at [735.0, 765.0] -> relabeling all to 735kV
Converting bridge transformer 'transf_55_3' (132-133) into an equivalent Line...

Assigning every non-backbone bus to its nearest backbone bus via graph shortest-path (cumulative real line length, multi-source Dijkstra)...

Aggregating loads onto their nearest backbone bus...
  647 original loads -> 58 aggregate loads on backbone buses

Aggregating generators onto (nearest backbone bus, carrier), using actual solved dispatch (not a re-derived capacity-factor ceiling)...
  262 original generators -> 82 aggregate generators

Ag

First line:  
Description: Builds the reduced network by aggregating components.  
Input: network/networks_current/elec_solved.nc  
Output: network/networks_current/elec_735kv.nc  

Second line:  
Description: Conducts non-linear Newton Raphson power flow (AC).  
Input: network/networks_current/elec_735kv.nc  
Output: network/networks_current/elec_735kv_pf.nc  

**Non-linear Newton Raphson power flow**

Slides 90-123
https://web.engr.oregonstate.edu/~webbky/ESE470_files/Section%205%20Power%20Flow.pdf

Each bus has four variables:
- $V_k$: voltage magnitude
- $\delta_k$: voltage phase
- $P_k$: real power 
- $Q_k$: reactive power
Power: difference between generated power and load power at node

For each bus, two of these variables are known and two must be calculated:
- Slack bus (reference): $V_1 = 1$, $\delta_1 = 1$
- Load bus (only load connections): $P_k$ and $Q_k$ are known
- Voltage-controlled bus (PV, connected to generators, with shunt reactive compensation): $P_k$ and $V_k$ are known.

Using:
$$I = YV \implies I_k = \sum_{n=1}^{N} Y_{kn}V_n$$
and 
$$S_k = P_k + jQ_k = V_kI_k^*$$
We obtain:
$$P_k = V_k\sum_{n=1}^{N}|Y_{kn}|V_ncos(\delta_k - \delta_n - \theta_{kn})$$
and 
$$Q_k = V_k\sum_{n=1}^{N}|Y_{kn}V_nsin(\delta_k - \delta_n - \theta_{kn})$$

The nonlinear system is solved using the iterative Newton-Raphson numerical method.

We have:
$$y=f(x)$$
where y is the power at each bus. We wish to find the voltages given an output power.

Step 1:
We may calculate the per-length impedence of each transmission line, and therefore find the admittance matrix.

Step 2:
We may define:
$$y = \begin{bmatrix} P \\ Q \end{bmatrix}$$
The values of which are given by PV bus (real power only) and PQ bus (real and reactive).

We may also define our unknown:
$$x = \left[\begin{array}{c} \delta \\ V \end{array}\right]$$
and initialize all unknown voltage phasors to:
$$V_k = 1 \angle 0^\circ$$

Step 3: define iteration parameters (max iterations, maximum error before convergence, ...).

Step 4: Begin iterating.  
Calculate $I_0$ from admittance. Plug to find $S_0$, $P_0$ and $Q_0$.  
Calculate power mismatch by comparing calculated values with previous values (for first iteration, set values).

Step 5: Set next values.  
We have the following relation:
$$\Delta y_0 = J_0 \Delta x_0$$
$J_0$ can be calculated (partial derivatives of known values by unknown values). Use Gauss elimination to find $\Delta x_0$.

Step 6: Update.  
We have:
$$x_1 = x_0 + \Delta x_0$$

Step 7: convergence check
$$\epsilon_0 = max|\frac{y_k - f_k(x)}{y_k}|



In [ ]:
# Circuit-count correction -> AC PF 110/168
%cd ~/Documents/McGill/capstone/quebec-pypsa-earth #replace with path to your repo!
!python network/correct_sending_end_circuits.py --network network/networks_current/elec_735kv.nc --in-place
!python network/run_pf.py --network network/networks_current/elec_735kv.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

/home/etienne/Documents/McGill/capstone/quebec-pypsa-earth
Loading network: network/networks_current/elec_735kv.nc
INFO:pypsa.io:Imported network elec_735kv.nc has buses, carriers, generators, lines, loads, storage_units
Corrected circuit count on 7 sending-end lines:
      num_parallel_old  r_old  x_old  b_old  s_nom_old  num_parallel_new  r_new   x_new  b_new  s_nom_new
Line                                                                                                     
2408               3.0   1.67  16.52   0.01   15728.35               3.0   1.67   55.07   0.01   15728.35
162                3.0   1.52  15.07   0.01   17046.58               3.0   1.52   50.22   0.01   17046.58
3756               3.0   2.08  68.48   0.01   13140.49               3.0   2.08   68.48   0.01   13140.49
1049               1.5   3.76  37.18   0.00    7119.33               1.5   3.76  123.94   0.00    7119.33
1448               1.5   3.78  37.47   0.00    7074.33               1.5   3.78  124.90   0.00 

First line:  
Description: Corrects the OSM modelling of specific lines in the 735 kV network.  
Input: network/networks_current/elec_735kv.nc --> reduced 735kV network  
Output: network/networks_current/elec_735kv.nc --> fixed reduced network  

Second line:  
Description: Runs AC power flow (described above) on the fixed network.
Input: network/networks_current/elec_735kv.nc --> fixed reduced network
Output: network/networks_current/elec_735kv_pf.nc --> solved, fixed, reduced network

In [ ]:
# 70% series compensation -> AC PF 168/168, 0.904-1.123 pu
%cd ~/Documents/McGill/capstone/quebec-pypsa-earth #replace with path to your repo!

!python network/apply_series_compensation.py --network network/networks_current/elec_735kv.nc --in-place
!python network/run_pf.py --network network/networks_current/elec_735kv.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

First line:  
Description: Applies 70% series compensation to the fixed network from above.  
Input: network/networks_current/elec_735kv.nc --> fixed reduced network  
Output: network/networks_current/elec_735kv.nc --> fixed reduced network with applied compensation  

Second line:  
Description: Runs power flow (described above) on the compensated network  
Input: network/networks_current/elec_735kv.nc --> fixed reduced network with applied compensation  
Output: network/networks_current/elec_735kv_pf.nc --> power flow applied to compensated network  

**70% series compensation**  

In general, we have:
$$P_1 = \frac{V_SV_R}{X_L} sin \delta$$
where $P_1$ is the power transferred over a line, $V_S$ is the voltage at the sending end, $V_R$ is the voltage at the recevining end, $X_L$ is the inductive reactance of the line and $\delta$ is the phase angle between sender and receiver voltages.  

Adding series compensation means plugging a capacitor in series with the line. Since capacitive reactance is negative, this gives:

$$X_{total} = X_L - X_C$$

Comparing power before and after compensation, we obtain:

$$\frac{P_1}{P_2} = \frac{1}{1-\frac{X_C}{X_L}}$$

where $\frac{X_C}{X_L}$ is the compensation factor (0.7).  

Advantages:
- Improves system stability. System stability is characterised by the phase angle $\delta$. This angle defines how much effort it will take for the line to transmit power. Generally, we consider that the line has a maximum transfer at $\delta = 90^\circ$. Lowering the phase angle means lines are less pushed to their limits.

- Also, since $Z = R + jX$, lowering X means lowering the series impedance of the line. This lowers the voltage variation across the line. For our project, the acceptance band for voltage is 0.95-1.05 pu.

https://circuitglobe.com/series-compensation.html



In [ ]:
# Shunt reactors and capacitors -> AC PF 168/168, 0.966-1.059 pu
%cd ~/Documents/McGill/capstone/quebec-pypsa-earth #replace with path to your repo!

!python network/add_shunt_impedance.py --network network/networks_current/elec_735kv.nc --output network/networks_current/elec_735kv_shunt.nc
!python network/run_pf.py --network network/networks_current/elec_735kv_shunt.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

First line:  
Description: Add shunt impedance to compensated network.  
Input: network/networks_current/elec_735kv.nc --> compensated, fixed, reduced network  
Output: network/networks_current/elec_735kv_shunt.nc --> network with applied shunts  

Second line:  
Description: Runs power flow (described above) on the shunted network.  
Input: network/networks_current/elec_735kv_shunt.nc  
Output: network/networks_current/elec_735kv_shunt_pf.nc  

**Shunt impedance**

Ferranti effect: on a long line with low load, voltage at the receiving end can be significantly larger than that at the sending end. Remember, we want to limit this variation in voltage.  
The line acts as a capacitor between other lines and the ground. When low load is used, then current is also lower, but the capacitor voltage does not change. If inductive and capacitive effects are not balance, it can lead to voltage increases or decreases. Capacitive current is still flowing (voltage is charging and discharging), which increases the voltage at the receiving end.  

Adding a shunt in parallel between a bus and ground can increase or decrease voltage (depending on the type, reactor absorbs reactive power and lowers voltage, shunt capacitor bank supplies reactive power and increases voltage).  

We have:
$$\delta V = \frac{RP + XQ}{V}$$
On high voltage lines, reactive impedance (used to move power) tends to dominate over real impedance (turns electrical energy into work).

https://www.theelectricalguyacademy.com/articles/why-are-shunt-reactors-used-in-power-systems

In [ ]:
# Comparison table of the three reactive-support cases
%cd ~/Documents/McGill/capstone/quebec-pypsa-earth #replace with path to your repo!

!python network/summarize_735kv_acpf.py

First line:  
Description: Summarises convergence of:  
- no reactive support network
- series compensation
- series compensation + shunts